In [ ]:
#Already done, output of this cell already available in dataset_big
# https://edgar.jrc.ec.europa.eu/dataset_ap81#p3
#Script to average two annual NetCDF files for NOx and two for CO, clip to Sub-Saharan Africa, and output two GeoTIFF files.
"""
import numpy as np
import xarray as xr
import rioxarray

def main():
    # Input files with absolute paths
    nox1 = r"C:/Users/matti/Desktop/nox_21.nc"
    nox2 = r"C:/Users/matti/Desktop/nox_22.nc"
    co1 = r"C:/Users/matti/Desktop/co_21.nc"
    co2 = r"C:/Users/matti/Desktop/co_22.nc"
    
    # Define bounding box for Sub-Saharan Africa
    lon_min, lon_max = -20, 55
    lat_min, lat_max = -35, 20
    
    # Process NOx
    nox_files = [nox1, nox2]
    nox_da = average_pollutant(nox_files, lon_min, lon_max, lat_min, lat_max)
    # Process CO
    co_files = [co1, co2]
    co_da = average_pollutant(co_files, lon_min, lon_max, lat_min, lat_max)
    
    # Write GeoTIFFs to desktop
    nox_da.rio.to_raster("dataset_big/NOx_SSA_avg.tif")
    co_da.rio.to_raster("dataset_big/CO_SSA_avg.tif")
    print("Output files written: NOx_SSA_avg.tif, CO_SSA_avg.tif")

def average_pollutant(files, lon_min, lon_max, lat_min, lat_max):
    #Read two NetCDF files, clip, average pixel-wise, return xarray DataArray.
    datasets = []
    for f in files:
        ds = xr.open_dataset(f)
        # Identify the data variable (first variable with lat/lon dims)
        data_var = None
        for var in ds.data_vars:
            if 'lat' in ds[var].dims and 'lon' in ds[var].dims:
                data_var = var
                break
        if data_var is None:
            raise ValueError(f"No variable with lat/lon dimensions in {f}")
        da = ds[data_var]
        # If there's a time dimension, average over it (assume annual file may have time)
        if 'time' in da.dims:
            da = da.mean(dim='time', keep_attrs=True)
        # Assign CRS (WGS84) for rioxarray
        da = da.rio.write_crs("EPSG:4326")
        # Clip to bounding box
        da = da.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
        datasets.append(da)
    # Pixel-wise average of the two files
    avg_da = (datasets[0] + datasets[1]) / 2.0
    # Preserve attributes
    avg_da.attrs = datasets[0].attrs
    return avg_da

if __name__ == "__main__":
    main()


"""

In [ ]:
"""
Script to compute weighted average of NOx and CO GeoTIFFs.
Reads from dataset_big/NOx_SSA_avg.tif and dataset_big/CO_SSA_avg.tif,
applies user-defined weights, and writes dataset_big/vehicles_emission_SSA.tif.
"""

import numpy as np
import rioxarray

def main():

    normalize = True          # to asses for difference in scale between NOx and CO
    norm_method = "minmax"    # option: "minmax", "zscore", "none"

    # User can change these weights at the top of the script
    weight_nox = 0.6
    weight_co = 0.4

    # Input file paths (relative to current working directory)
    nox_path = "dataset_big/NOx_SSA_avg.tif"
    co_path = "dataset_big/CO_SSA_avg.tif"
    output_path = "dataset_big/vehicles_emission_SSA.tif"

    # Read the GeoTIFFs with rioxarray
    nox_da = rioxarray.open_rasterio(nox_path)
    co_da = rioxarray.open_rasterio(co_path)

    #normalize
    if normalize:
        if norm_method == "minmax":
            nox_min = nox_da.min().item()
            nox_max = nox_da.max().item()
            co_min = co_da.min().item()
            co_max = co_da.max().item()
            nox_da = (nox_da - nox_min) / (nox_max - nox_min)
            co_da = (co_da - co_min) / (co_max - co_min)
        elif norm_method == "zscore":
            nox_mean = nox_da.mean().item()
            nox_std = nox_da.std().item()
            co_mean = co_da.mean().item()
            co_std = co_da.std().item()
            nox_da = (nox_da - nox_mean) / nox_std
            co_da = (co_da - co_mean) / co_std
        elif norm_method == "none":
            pass
        else:
            raise ValueError(f"Unknown normalization method: {norm_method}")

    # Compute weighted average
    weighted_avg = (weight_nox * nox_da) + (weight_co * co_da)

    # Preserve metadata (optional)
    weighted_avg.attrs = nox_da.attrs
    weighted_avg.rio.write_crs(nox_da.rio.crs, inplace=True)

    # Write output GeoTIFF
    weighted_avg.rio.to_raster(output_path)

    print(f"Output written to {output_path}")
    print(f"Weights: NOx = {weight_nox}, CO = {weight_co}")

if __name__ == "__main__":
    main()

In [ ]:
"""
Script to clip the vehicle emission raster to Nigeria, reproject to Web Mercator (EPSG:3857),
and resample to 1 km resolution.
"""

import geopandas as gpd
import rioxarray
import numpy as np
import xarray as xr
from shapely.geometry import mapping
from rasterio.warp import Resampling

def main():
    
    # User parameters (can be changed at the top)
    
    raster_path = "dataset_big/vehicles_emission_SSA.tif"
    geojson_path = "dataset_big/Country_boundaries.geojson"
    output_path = "dataset_big/emission_nigeria_3857.tif"
    target_resolution = 1000  # meters in EPSG:3857 (1 km)
    resample_method = Resampling.nearest
    target_crs = "EPSG:3857"
    
  
    # 1. Read the raster
   
    print("Reading emission raster...")
    da = rioxarray.open_rasterio(raster_path)
    original_crs = da.rio.crs
    print(f"Original CRS: {original_crs}")
    print(f"Original resolution: {da.rio.resolution()}")
    
    # 2. Read the GeoJSON and extract Nigeria polygon
    print(f"Reading GeoJSON from {geojson_path}...")
    gdf = gpd.read_file(geojson_path)
    print(f"GeoJSON columns: {gdf.columns.tolist()}")
    
    # Find the column that contains country names (try common names)
    name_col = None
    for col in ["NAME", "name", "country", "Country", "ADMIN", "admin"]:
        if col in gdf.columns:
            name_col = col
            break
    if name_col is None:
        # If not found, assume the first geometry is Nigeria (fallback)
        print("Warning: Could not find a country name column. Using first geometry.")
        nigeria_geom = gdf.geometry.iloc[0]
    else:
        nigeria_mask = gdf[name_col].str.strip().str.lower() == "nigeria"
        if not nigeria_mask.any():
            print(f"Nigeria not found in column '{name_col}'. Available values: {gdf[name_col].unique()[:10]}")
            raise ValueError("Nigeria not found in GeoJSON.")
        nigeria_geom = gdf.loc[nigeria_mask, "geometry"].iloc[0]
    
    print(f"Nigeria geometry extracted.")
    
    # 3. Reproject raster to target CRS (EPSG:3857) and clip to Nigeria
    
    print(f"Reprojecting raster to {target_crs}...")
    da_3857 = da.rio.reproject(target_crs)
    
    print("Clipping raster to Nigeria geometry...")
    # Clip using the geometry (transform to same CRS as raster)
    nigeria_geom_3857 = gpd.GeoSeries([nigeria_geom], crs=gdf.crs).to_crs(target_crs).iloc[0]
    da_clipped = da_3857.rio.clip([mapping(nigeria_geom_3857)], drop=True)
    
    
    # 4. Resample to 1 km resolution (nearest neighbor)

    print(f"Resampling to {target_resolution} m resolution using method '{resample_method}'...")
    # Create a new grid with pixel size = target_resolution
    # Use rioxarray.reproject with target_resolution (only if we want to force that resolution)
    # We'll simply set the resolution and let rioxarray handle it.
    da_resampled = da_clipped.rio.reproject(
        target_crs,
        resolution=target_resolution,
        resampling=resample_method
    )
    
    # 5. Write output

    print(f"Writing output to {output_path}...")
    da_resampled.rio.to_raster(output_path, compress="DEFLATE")
    print("Done.")
    
if __name__ == "__main__":
    main()